In [92]:
import pandas as pd
import numpy as np


def create_ebird_df(target_route, target_year):
    routes_df = pd.read_csv("data/routes.csv")
    trips_df = pd.read_csv("data/trips.csv")
    trips_df["id"] = trips_df.apply(lambda x: f"{int(x["route_id"])}-{int(x["yyyy"])}", axis=1)
    sites_df = pd.read_csv("data/sites.csv")
    sites_df["id"] = sites_df.apply(lambda x: f"{int(x["route_id"])}-{int(x["stop_number"])}", axis=1)
    trip_id = f"{target_route}-{target_year}"
    raw_data = pd.read_csv(f"data/raw_counts/{target_route}/{target_year}.csv", names=[f"col{x}" for x in range(1, 8)])

    # convert data into a better format
    data = dict(species=[], stop_number=[], count=[])

    recording = False
    sites = None
    for index, row in raw_data.iterrows():
        # if the first item in row is "Species" we are starting a new block.
        if row.iloc[0] == "Number of Vehicles":
            recording = False
        elif row.iloc[0] == "Species":
            recording = True
            sites = [int(x) for x in row.iloc[1:6]]
            continue
        if recording and sites:
            # each col should be an observation in the new table
            species = row.iloc[0]
            tallies = row.iloc[1:6]
            for t, site in zip(tallies, sites):
                if pd.notna(t):
                    data["species"].append(species)
                    data["stop_number"].append(site)
                    data["count"].append(t)

    df = pd.DataFrame(data)
    df["route_id"] = target_route
    df["trip_id"] = trip_id

    # merge route information based on the route id
    df = df.merge(routes_df, left_on="route_id", right_on="id")
    # merge site information based on the trip id
    df["site_id"] = df.apply(lambda x: f"{int(x["route_id"])}-{int(x["stop_number"])}", axis=1)
    df = df.merge(sites_df, left_on="site_id", right_on="id")
    # merge trip information based on the trip id
    df = df.merge(trips_df, left_on="trip_id", right_on="id")

    ebird_data = {
        "Common Name": [],
        "Genus": [],
        "Species": [],
        "Number": [],
        "Species Comments": [],
        "Location Name": [],
        "Latitude": [],
        "Longitude": [],
        "Date": [],
        "Start Time": [],
        "State/Province": [],
        "Country Code": [],
        "Protocol": [],
        "Number of Observers": [],
        "Duration": [],
        "All observations reported?": [],
        "Effort Distance Miles": [],
        "Effort area acres": [],
        "Submission Comments": [],
    }

    for index, row in df.iterrows():
        ebird_data["Common Name"].append(row["species"])
        ebird_data["Genus"].append("")
        ebird_data["Species"].append("")
        ebird_data["Number"].append(row["count"])
        ebird_data["Species Comments"].append(np.nan)
        ebird_data["Location Name"].append(f"BBS {row["name"]} ({row["route_id"]}) site {row["stop_number_x"]}" )
        ebird_data["Latitude"].append(row["latitude"])
        ebird_data["Longitude"].append(row["longitude"])
        ebird_data["Date"].append(f"{str(row["mm"]).rjust(2, "0")}/{str(row["dd"]).rjust(2, "0")}/{row["yyyy"]}")
        ebird_data["Start Time"].append(f"{str(row["start_hh"]).rjust(2, "0")}:{str(row["start_mm"]).rjust(2, "0")}")
        ebird_data["State/Province"].append(row["province"])
        ebird_data["Country Code"].append(row["country"])
        ebird_data["Protocol"].append("area")
        ebird_data["Number of Observers"].append(row["no_observers"])
        ebird_data["Duration"].append(3)
        ebird_data["All observations reported?"].append("Y")
        ebird_data["Effort Distance Miles"].append(np.nan)
        ebird_data["Effort area acres"].append(124)  # 400m radius
        ebird_data["Submission Comments"].append("Data submitted to USGS")

    ebird_df = pd.DataFrame(ebird_data)
    ebird_df.to_csv(f"ebird_output/{trip_id}-checklist.csv", header=False, index=False)

create_ebird_df(65023, 2026)
create_ebird_df(56042, 2026)
create_ebird_df(65023, 2025)
create_ebird_df(65023, 2024)
create_ebird_df(65023, 2023)
create_ebird_df(65023, 2022)
create_ebird_df(65023, 2019)
create_ebird_df(65023, 2018)
create_ebird_df(65023, 2017)
create_ebird_df(65023, 2016)
create_ebird_df(65023, 2015)
